In [38]:
using LinearAlgebra
#using Plots
using LaTeXStrings
using SparseArrays
using Printf
using GLMakie
using Statistics

In [2]:
function dif_1_centrada(f, vec_x, h)
    dif_vec = []
    for j in 2:length(vec_x)-1
        derivada = (f(vec_x[j+1]) - f(vec_x[j-1]))/(2*h)
        append!(dif_vec, derivada)
    end

    return dif_vec
end

dif_1_centrada (generic function with 1 method)

In [3]:
function dif_2_centrada(f, vec_x, h)
    dif_vec = []
    for j in 2:length(vec_x)-1
        derivada = (f(vec_x[j+1]) -2*(f(vec_x[j])) + f(vec_x[j-1]))/(h^2)
        append!(dif_vec, derivada)
    end

    return dif_vec
end

dif_2_centrada (generic function with 1 method)

In [4]:
function eq_discretizada(f, k, h, init_sol, x_range, t_n, n_steps, params, tol=10e-6)
    α, u = params
    v_f = init_sol[end, 1]

    dif_1 = dif_1_centrada(identity, init_sol, h)
    dif_2 = dif_2_centrada(identity, init_sol, h)

    #display(dif_1)
    #display(dif_2)
    
    sol = []
    for j in 1:n_steps
            
        if j == n_steps
           append!(sol, v_f)
           continue
        end

        r = x_range[j]
        if(r == 0)
            v = init_sol[j] + k*(2*α/u)*(2*(init_sol[j+1] - init_sol[j])/h^2)
        else
            v = init_sol[j] + k*(α/u)*(dif_2[j-1] + (1/r) * dif_1[j-1] - f(t_n, init_sol[j]))
        end
        
        append!(sol, v)        
    end

    #display(sol)
    # Iteração de Convergência aqui, se implícito    
    return sol
end

eq_discretizada (generic function with 2 methods)

In [5]:
function main(f, t_interval, t_steps, x_interval, x_steps, init_sol, params, metodo)
    t_0, t_f = t_interval
    k = (t_f - t_0)/t_steps

    x_0, x_f = x_interval
    h = (x_f - x_0)/x_steps

    vecs_sol = copy(init_sol)
    x_range = x_0:h:x_f
    
    for t_n in (t_0+k):k:t_f
        sol = metodo(f, k, h, init_sol, x_range, t_n, x_steps, params)
        vecs_sol = hcat(vecs_sol, sol)

        init_sol = hcat(init_sol, sol)[:, 2:end]        
    end

    return vecs_sol
end

main (generic function with 1 method)

In [86]:
X = 5
a = 0.1 * 10e-2

intervalo_r = [0, a]
intervalo_x = [0, X]

passos_r = 20
passos_x = 10000

f = (t, x) -> 0.0
t_i = 90.0
t_w = 0.0
init_sol = vcat(repeat([t_i], passos_r - 1), [t_w])

α = 1.5 * 10e-7
u = 1 * 10e-2
params = α, u

result = main(f, intervalo_x, passos_x, intervalo_r, passos_r, init_sol, params, eq_discretizada)

20×10001 Matrix{Any}:
 90.0  90.0    90.0     90.0     90.0     …  1.18616   1.18559    1.18503
 90.0  90.0    90.0     90.0     90.0        1.18142   1.18085    1.18029
 90.0  90.0    90.0     90.0     90.0        1.16724   1.16668    1.16612
 90.0  90.0    90.0     90.0     90.0        1.1438    1.14325    1.1427
 90.0  90.0    90.0     90.0     90.0        1.11137   1.11084    1.11031
 90.0  90.0    90.0     90.0     90.0     …  1.07035   1.06984    1.06932
 90.0  90.0    90.0     90.0     90.0        1.02122   1.02073    1.02024
 90.0  90.0    90.0     90.0     90.0        0.964577  0.964114   0.963652
 90.0  90.0    90.0     90.0     90.0        0.901083  0.900651   0.900219
 90.0  90.0    90.0     90.0     90.0        0.831495  0.831096   0.830697
 90.0  90.0    90.0     90.0     90.0     …  0.756631  0.756268   0.755906
 90.0  90.0    90.0     90.0     90.0        0.677372  0.677047   0.676722
 90.0  90.0    90.0     90.0     90.0        0.594642  0.594356   0.594071
 90.0  90.0

In [87]:
h = (intervalo_r[2] - intervalo_r[1])/passos_r

r_range = collect(range(intervalo_r[1], intervalo_r[2]-h, passos_r-1))
x_range = collect(range(intervalo_x[1], intervalo_x[2], passos_x))

X = [x for x = x_range for _ = r_range]
R = [r for _ = x_range for r = r_range]
vec_res = vec(result[1:end-1, 1:end-1])

# Interactive surface plot
fig, ax, plt = surface(R, X, vec_res, 
    axis=(; type=Axis3, 
          title="Interactive Surface",
          xlabel="R", ylabel="X", zlabel="T"),
    colormap=:viridis
)

# Display with interactivity
display(fig)

GLMakie.Screen(...)

In [70]:
k = (intervalo_x[2] - intervalo_x[1])/passos_x
nu = (2*a)/(-mean(result[:, end]))*(3*result[end, end] - 4*result[end-1, end] + result[end-2, end])/(2*k)

3.6306239102167885